# In this project, we plan to design a model that can predict whether the price of a product has been mentioned in a comment. So let's get started and dig through the comments to get to the price!

- One of the most common scenarios is that a company, website or application intends to analyze user comments about its product or products in different ways to determine its business strategy based on the results obtained.
- Since the number of received comments is very large in large companies and human resources are practically unable to check all of them, machine learning is used to analyze and interpret these data.
- For example, one of the practical problems in this field is determined in such a way that the machine is required to study a comment and predict whether this comment has a positive feeling or not.
- For example, in relation to a product, whether the user was satisfied with the purchase and the received product or not.

# Load data

|column|description|
|:------:|:---:|
|<code>comment</code>|comment text|
|<code>price_value</code>|Did it talk about the price? (<code>1</code>) or no (<code>0</code>)||

The data set that we are going to use is related to the comments registered on the Digikala website. Each of the comments in this collection is tagged in terms of whether or not the price of the product is mentioned in it.
There are 40,000 comments in the dataset.

In [66]:
import pandas as pd

df = pd.read_csv('/content/digikala_comments.csv')
df.head()

,comment,price_value
0,قیمت مناسب وکیفیت خوب پیشنهادمیکنم حتما خرید کنید,1
1,به اندازه یک میلیمتر دورتادور گوشی خالی میماند...,0
2,از همه نظر عالی و یک خرید خوب در قیمت حدود۴۰ ...,1
3,فقط یک بار هر یک ربع ساعت 1 درصد شارژ کرد بعدش...,0
4,قیمت این کالا خیلی تغییر میکنه . من خریدم چندر...,1


# Preprocessing

In [70]:
from hazm import Normalizer, WordTokenizer, Lemmatizer, stopwords_list

punctuation_marks = "،.!؟؛:\"'()[]-—"

normalizer = Normalizer()
lemmatizer = Lemmatizer()
word_tokenizer = WordTokenizer(separate_emoji=True, replace_numbers=True)


def preprocessor(comment):
    if not isinstance(comment, str):
        return ""

    comment = normalizer.normalize(comment)
    word_list = word_tokenizer.tokenize(comment)

    lemmatize_word = list(map(lemmatizer.lemmatize, word_list))
    filtered_word = list(filter(lambda x: x not in stopwords_list() and not x.isdigit() and x not in punctuation_marks, lemmatize_word))

    return ' '.join(filtered_word)


preprocessor('از همه نظر عالی و یک خریدی خوب، در  قیمت حدود۴۰ تومن')

'خرید#خر قیمت NUM تومن'

In [72]:
comments = df['comment'].apply(preprocessor)

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib


vectorizer = TfidfVectorizer()
vectorizer.fit(comments)

['vectorizer.pkl']

In [75]:
from sklearn.model_selection import train_test_split


X = vectorizer.transform(comments)
y = df['price_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, random_state=42, test_size=0.2)

# Modeling

In [88]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred) * 100, '%')

accuracy: 90.03750000000001 %


In [91]:
model.predict(vectorizer.transform([preprocessor('در مورد قیمت حرف میزنم!!!')]))[0]

1

In [93]:
model.predict(vectorizer.transform([preprocessor('فقط نظر راجب کیفیت دارم')]))[0]

0